In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

In [ ]:
import pandas as pd


df = pd.read_csv("/run/media/victor/pessoal/mestrado/codigo/database_scripts/V2_leiomyoma_language_clean.csv")

In [ ]:

from nltk.corpus import wordnet
import re
from tqdm import tqdm
from nltk.stem import WordNetLemmatizer
from nltk.corpus.reader.wordnet import WordNetError

def is_synset_name_valid(name):

    try:
        synset = wordnet.synset(name)
        return True
    
    except WordNetError:
        return False

def is_animal(word: str, animal_synset, human_synset, lemmatizer) -> bool:
    lemma = lemmatizer.lemmatize(word)

    try:
        synset = wordnet.synset(lemma + '.n.01')
    except:
        return False

    if not synset:
        return False
    
    if wordnet.synset('fetus.n.01') == synset or wordnet.synset('placental.n.01') == synset:
        return False
      

    if (not is_synset_name_valid(lemma + '.a.01')) and (not human_synset in synset.lowest_common_hypernyms(human_synset)):
        
        if (animal_synset in synset.lowest_common_hypernyms(animal_synset)):
            return True
        
            
    return False

def is_animal_adjective(word: str, animal_synset, human_synset, lemmatizer) -> bool:
    
    lemma = lemmatizer.lemmatize(word)

    if lemma == "bovine" or lemma == "canine":
        return True

    try:
        synset = wordnet.synset(lemma + '.a.01')
    except:
        return False
    
    for lemma in synset.lemmas():
        for pertainym in lemma.pertainyms():
            if is_animal(pertainym.name(), animal_synset, human_synset, lemmatizer):
                return True

                        
    return False

def extract_animals(text, max_ngram, animal_synset, human_synset, lemmatizer):

    words = re.findall(r'\b\w+(?:-\w+)*\b', text.lower())
    animals_found = set()
    
    for n in range(max_ngram, 0, -1):
        for i in range(len(words) - n + 1):
            phrase = '_'.join(words[i:i + n])
            
            if is_animal(phrase, animal_synset, human_synset, lemmatizer):
                animals_found.add(phrase)
    
    return list(animals_found)

def extract_adjectives(text: str, max_ngram, animal_synset, human_synset, lemmatizer):
    words = re.findall(r'\b\w+(?:-\w+)*\b', text.lower())
    animals_adjetives_found = set()
    
    # Invert range to check longer n-grams first
    for n in range(max_ngram, 0, -1):
        for i in range(len(words) - n + 1):
            phrase = '_'.join(words[i:i + n])
            
            if is_animal_adjective(phrase, animal_synset, human_synset, lemmatizer):
                animals_adjetives_found.add(phrase)
                        
    return list(animals_adjetives_found)


max_ngram = 7
animal_synset = wordnet.synset('animal.n.01')
human_synset = wordnet.synset('person.n.01')
lemmatizer = WordNetLemmatizer()

adjectives = []
animals = []

for index, row in tqdm(df.iterrows(), total=len(df)):
 
    caption = row['caption']

    identified_animals = extract_animals(caption, max_ngram, animal_synset, human_synset, lemmatizer)
    identified_animal_adjectives = extract_adjectives(caption, max_ngram, animal_synset, human_synset, lemmatizer)

    adjectives.extend(identified_animal_adjectives)
    animals.extend(identified_animals)

    df.at[index, "animals"] = ', '.join(identified_animals) if len(identified_animals) > 0 else None
    df.at[index, 'animal_adjectives'] = ', '.join(identified_animal_adjectives) if len(identified_animal_adjectives) > 0 else None



In [ ]:
pd.DataFrame(adjectives).value_counts()

In [ ]:
pd.DataFrame(animals).value_counts()

In [ ]:
df

In [ ]:
filtered_df = df[df["animals"].isna() & df["animal_adjectives"].isna()]

In [ ]:
filtered_df

In [ ]:
filtered_df = filtered_df.drop(columns=["is_histopathology", "animals", "animal_adjectives"])

In [ ]:
filtered_df.to_csv("V2_leiomioma_animal_clean.csv", index=False)

In [ ]:
   
import shutil

dst_path = "/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leiomioma_animal_clean/"

for index, row in filtered_df.iterrows():
    shutil.copy(row["image_path"], dst_path + row["article_id"] + "-" + row["image_path"].split("/")[-1])   
